# 8장 — 공은 어디서 멈출까: 평형과 안정성

웹 챕터(8장)와 **같은 모형, 같은 수치**를 파이썬으로 계산합니다:
위상선으로 평형 찾기, 기울기로 안정성 판정, 임계 질량(문턱), 그리고 두 저량(모멘텀)의 나선과 고유값.

> 🤖 **AI 튜터와 함께 공부하세요**: 아래 셀을 실행하면 화면 오른쪽 아래에 튜터 버튼이 뜹니다.
> 로컬 Ollama 가 꺼져 있어도 노트북 학습에는 지장이 없습니다.

In [ ]:
from tutor import chat
chat('ch8')  # 이 챕터를 아는 AI 튜터 (로컬 Ollama — 없어도 학습엔 지장 없음)

## 1. 위상선: 순유량이 0인 곳

| 모형 | 순유량 $f(S)$ | 평형 |
|---|---|---|
| 로지스틱 | $rS(1 - S/K)$ | 0(불안정), K(안정) |
| 임계 질량 | $rS(S/A - 1)(1 - S/K)$ | 0(안정), A(불안정, 문턱), K(안정) |

$r = 0.5$, $A = 200$, $K = 1{,}000$. 평형 근처의 기울기 $f'(S^*)$가 음수면 안정, 양수면 불안정.

In [ ]:
r, A, K = 0.5, 200, 1000
logistic = lambda S: r * S * (1 - S / K)
allee = lambda S: r * S * (S / A - 1) * (1 - S / K)

def slope(f, x, h=1e-6):
    return (f(x + h) - f(x - h)) / (2 * h)

for name, f, eqs in [('로지스틱', logistic, [0, K]), ('임계 질량', allee, [0, A, K])]:
    for e in eqs:
        d = slope(f, e)
        print(f'{name:6s}  평형 S* = {e:5d}:  f(S*) = {f(e):.1e},  f\'(S*) = {d:+.3f}  →  {"안정" if d < 0 else "불안정"}' + (f' (시간 상수 {1 / abs(d):.1f}개월)' if d < 0 else ''))

assert abs(slope(logistic, 0) - 0.5) < 1e-6 and abs(slope(logistic, K) + 0.5) < 1e-6
assert slope(allee, 0) < 0 < slope(allee, A) and slope(allee, K) < 0
print('✅ 로지스틱: 0 불안정·K 안정(시간 상수 2개월) · 임계 질량: 0 안정·200 불안정·1,000 안정')

## 2. 출발점에 따라 갈리는 결과 (임계 질량)

오일러법(DT = 1/16개월)으로 36개월을 계산합니다(웹 sim.ts phaseRun 과 같은 계산).

In [ ]:
def run(f, S0, T=36, dt=1 / 16):
    S = S0
    for _ in range(round(T / dt)):
        S += f(S) * dt
    return S

for S0 in [150, 190, 199, 201, 210, 250, 400]:
    print(f'처음 {S0:3d}명 → 36개월 뒤 {run(allee, S0):7.1f}명')
assert run(allee, 150) < 1 and run(allee, 199) < 1          # 문턱 아래: 사라짐
assert abs(run(allee, 201) - K) < 1 and abs(run(allee, 250) - K) < 1   # 문턱 위: 1,000명
assert abs(run(logistic, 10) - K) < 1
print('✅ 200명(불안정 평형)이 문턱: 150명이면 사라지고 250명이면 1,000명까지')

## 3. 저량이 둘이면: 모멘텀의 나선과 고유값

곡률 2, 학습률 0.05(곱 0.1), 출발점 8. 속도 $v \leftarrow \beta v - 0.1x$, 위치 $x \leftarrow x + v$.
한 걸음은 (위치, 속도)에 행렬 $\begin{pmatrix} 0.9 & \beta \\ -0.1 & \beta \end{pmatrix}$ 를 곱하는 일이고(1부 7장),
고유값이 복소수면 나선, 크기가 1보다 작으면 결국 멈춥니다.

In [ ]:
import numpy as np

def heavy_ball(beta, x0=8.0, steps=60, a=0.1):
    x, v = x0, 0.0
    xs, vs = [x], [v]
    for _ in range(steps):
        v = beta * v - a * x
        x = x + v
        xs.append(x); vs.append(v)
    return xs, vs

for beta in [0.0, 0.3, 0.5, 0.7, 0.9]:
    M = np.array([[0.9, beta], [-0.1, beta]])
    ev = np.linalg.eigvals(M)
    xs, _ = heavy_ball(beta)
    over = max(0.0, -min(xs))
    kind = '나선' if abs(ev[0].imag) > 1e-12 else '곧장'
    print(f'β = {beta:.1f}: 고유값 {np.round(ev, 3)}  크기 {abs(ev[0]):.3f}  → {kind},  반대편으로 넘어간 최대 거리 {over:.2f}')

boundary = (1 - 0.1 ** 0.5) ** 2
print(f'나선이 시작되는 β: (1 − √0.1)² = {boundary:.3f}')
assert abs(boundary - 0.468) < 0.001
assert abs(abs(np.linalg.eigvals(np.array([[0.9, 0.9], [-0.1, 0.9]]))[0]) - 0.9 ** 0.5) < 1e-12   # 크기 = √β
assert round(-min(heavy_ball(0.9)[0]), 2) == 4.83
assert min(heavy_ball(0.3)[0]) >= 0                     # β = 0.3: 넘어가지 않음
print('✅ β > 약 0.468 이면 고유값이 복소수 → 나선 · β = 0.9 면 반대편으로 4.83까지')

In [ ]:
import matplotlib.pyplot as plt
plt.rcParams['font.family'] = 'Malgun Gothic'
plt.rcParams['axes.unicode_minus'] = False

fig, axes = plt.subplots(1, 3, figsize=(15, 4.2))
Ss = np.linspace(0, 1200, 241)
axes[0].plot(Ss, [allee(s) for s in Ss], color='#4f46e5', lw=2.2, label='임계 질량')
axes[0].plot(Ss, [logistic(s) for s in Ss], color='#94a3b8', lw=1.5, label='로지스틱')
axes[0].axhline(0, color='#64748b')
for e, stable in [(0, True), (A, False), (K, True)]:
    axes[0].plot(e, 0, 'o', ms=9, color='#059669' if stable else '#e11d48')
axes[0].set_xlabel('사용자 수 S'); axes[0].set_ylabel('순유량'); axes[0].set_title('위상선: 초록 안정, 빨강 불안정'); axes[0].legend(); axes[0].grid(alpha=0.3)
ts = np.arange(0, 36 + 1e-9, 1 / 16)
for S0, c in [(150, '#e11d48'), (199, '#fb7185'), (201, '#34d399'), (250, '#059669')]:
    S, path = S0, [S0]
    for _ in ts[1:]:
        S += allee(S) / 16; path.append(S)
    axes[1].plot(ts, path, color=c, lw=2, label=f'처음 {S0}명')
axes[1].axhline(A, ls='--', color='#e11d48', lw=1)
axes[1].set_xlabel('개월'); axes[1].set_title('문턱 200명을 넘느냐가 결과를 가른다'); axes[1].legend(fontsize=8); axes[1].grid(alpha=0.3)
for beta, c in [(0.3, '#c4b5fd'), (0.7, '#8b5cf6'), (0.9, '#4c1d95')]:
    xs, vs = heavy_ball(beta)
    axes[2].plot(xs, vs, marker='.', color=c, lw=1.2, label=f'β = {beta}')
axes[2].axhline(0, color='#94a3b8'); axes[2].axvline(0, color='#94a3b8')
axes[2].set_xlabel('위치 x'); axes[2].set_ylabel('속도 v'); axes[2].set_title('위상 평면: β가 크면 나선'); axes[2].legend(); axes[2].grid(alpha=0.3)
plt.tight_layout()
plt.show()

## 4. 오늘의 발견

- **평형** = 순유량이 0인 곳. **위상선**(저량 축 위에 평형과 움직이는 방향을 표시한 그림, 여기서는 가로 저량·세로 순유량 그래프)에서 순유량이 0인 점.
- 평형 근처의 기울기 $f'(S^*)$: 음수면 **안정**(시간 상수 $1/|f'|$로 돌아옴), 양수면 **불안정**(멀어짐).
- **임계 질량**(불안정 평형)이 있으면 출발점이 결과를 가른다: 150명 → 0명, 250명 → 1,000명.
- 저량이 둘(위치·속도)이면 지연 없이도 **나선**으로 출렁일 수 있다. 판정은 **고유값**(복소수면 나선, 크기 < 1이면 수렴).
- 학습이 멈추는 곳도 평형: 최저점 = 안정, 안장점 = 반쪽짜리 평형(8부 10장).

다음 장(9장 그림만 보면 틀린다)에서 인과지도를 제대로 그리는 규칙과 함정을 봅니다.
**웹 챕터로 돌아가 연습 10문제를 풀면 진도가 채워져요.**